### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
## Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

## Step 1: Split into sentences (improved: handle multiple sentences per line and clean up)
import re
print("🔍 Splitting text into sentences...")
print("Before Splitting:")
print("Original Text:", text)
sentences = re.split(r'(?<=[.!?])\s+', text.strip())
print("After Splitting: step 1")
print("Sentences:\n", sentences)
sentences = [s.strip() for s in sentences if s.strip()]
print("After Splitting: step 2")
print("Cleaned Sentences:\n", sentences)

if not sentences:
    print("No sentences found in the text.")
else:
    ## Step 2: Embed each sentence
    embeddings = model.encode(sentences)

    ## Step 3: Initialize parameters
    threshold = 0.7  # Similarity threshold for chunking
    chunks = []
    current_chunk = [sentences[0]]

    ## Step 4: Semantic grouping based on threshold
    for i in range(1, len(sentences)):
        sim_values = cosine_similarity([embeddings[i - 1]], [embeddings[i]])
        print(f"cosine_similarity of sentence {i}, {i + 1} : ", sim_values)
        sim = sim_values[0][0]
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]

    ## Append the last chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    ## Output the chunks
    print("\n📌 Semantic Chunks:")
    for idx, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {idx}:\n{chunk}")

🔍 Splitting text into sentences...
Before Splitting:
Original Text: 
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.

After Splitting: step 1
Sentences:
 ['LangChain is a framework for building applications with LLMs.', 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.', 'You can create chains, agents, memory, and retrievers.', 'The Eiffel Tower is located in Paris.', 'France is a popular tourist destination.']
After Splitting: step 2
Cleaned Sentences:
 ['LangChain is a framework for building applications with LLMs.', 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.', 'You can create chains, agents, memory, and retrievers.', 'The Eiffel Tower is located in Pari

In [3]:
import pandas as pd

print(f"\n📊 Embeddings shape: {embeddings.shape}")

## Compute pairwise cosine similarity matrix
similarity_matrix = cosine_similarity(embeddings)
print(similarity_matrix)

## Create pandas DataFrame for better visualization
sentence_labels = [f'S{i+1}' for i in range(len(sentences))]
df_similarity = pd.DataFrame(
    similarity_matrix,
    index=sentence_labels,
    columns=sentence_labels
)

print(f"\n🔗 Pairwise Cosine Similarity Matrix ({len(sentences)}x{len(sentences)}):")
print(df_similarity.round(2))  # Round to 2 decimal places for readability

## Optional: Show the actual sentences for reference
print("\n📝 Sentence Reference:")
for i, sent in enumerate(sentences):
    print(f"S{i+1}: {sent}")


📊 Embeddings shape: (5, 384)
[[ 1.          0.8263335   0.37107393 -0.04267288  0.01440363]
 [ 0.8263335   0.9999995   0.4556986  -0.04850001 -0.03608257]
 [ 0.37107393  0.4556986   1.          0.02249055 -0.02198501]
 [-0.04267288 -0.04850001  0.02249055  1.0000001   0.37755758]
 [ 0.01440363 -0.03608257 -0.02198501  0.37755758  1.        ]]

🔗 Pairwise Cosine Similarity Matrix (5x5):
      S1    S2    S3    S4    S5
S1  1.00  0.83  0.37 -0.04  0.01
S2  0.83  1.00  0.46 -0.05 -0.04
S3  0.37  0.46  1.00  0.02 -0.02
S4 -0.04 -0.05  0.02  1.00  0.38
S5  0.01 -0.04 -0.02  0.38  1.00

📝 Sentence Reference:
S1: LangChain is a framework for building applications with LLMs.
S2: Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
S3: You can create chains, agents, memory, and retrievers.
S4: The Eiffel Tower is located in Paris.
S5: France is a popular tourist destination.


In [4]:
## Compare with NLTK sentence tokenizer
import nltk
nltk.download('punkt', quiet=True)  # Download punkt tokenizer if not already downloaded

# Complex text with abbreviations, quotes, and multiple punctuation
complex_text = """
Dr. Smith went to the U.S.A. last year. He said, "Hello! How are you?" The meeting was at 3 p.m. in New York.
 I think it's great... don't you? Mrs. Johnson replied, "Yes, indeed!" What about tomorrow? Let's discuss it then.
"""

print("🔍 Complex Text:")
print(complex_text.strip())

# NLTK sentence tokenization
nltk_sentences = nltk.sent_tokenize(complex_text.strip())
print(f"\n📝 NLTK Sentences ({len(nltk_sentences)}):")
for i, sent in enumerate(nltk_sentences, 1):
    print(f"{i}. {sent}")

# Regex-based splitting (from our earlier method)
import re
regex_sentences = re.split(r'(?<=[.!?])\s+', complex_text.strip())
regex_sentences = [s.strip() for s in regex_sentences if s.strip()]
print(f"\n🔄 Regex Sentences ({len(regex_sentences)}):")
for i, sent in enumerate(regex_sentences, 1):
    print(f"{i}. {sent}")

print("\n📊 Comparison:")
print(f"NLTK found {len(nltk_sentences)} sentences")
print(f"Regex found {len(regex_sentences)} sentences")
print("NLTK handles complex cases better (e.g., abbreviations, quotes), while regex is simpler but may split incorrectly.")

🔍 Complex Text:
Dr. Smith went to the U.S.A. last year. He said, "Hello! How are you?" The meeting was at 3 p.m. in New York.
 I think it's great... don't you? Mrs. Johnson replied, "Yes, indeed!" What about tomorrow? Let's discuss it then.

📝 NLTK Sentences (8):
1. Dr. Smith went to the U.S.A. last year.
2. He said, "Hello!
3. How are you?"
4. The meeting was at 3 p.m. in New York.
5. I think it's great... don't you?
6. Mrs. Johnson replied, "Yes, indeed!"
7. What about tomorrow?
8. Let's discuss it then.

🔄 Regex Sentences (11):
1. Dr.
2. Smith went to the U.S.A.
3. last year.
4. He said, "Hello!
5. How are you?" The meeting was at 3 p.m.
6. in New York.
7. I think it's great...
8. don't you?
9. Mrs.
10. Johnson replied, "Yes, indeed!" What about tomorrow?
11. Let's discuss it then.

📊 Comparison:
NLTK found 8 sentences
Regex found 11 sentences
NLTK handles complex cases better (e.g., abbreviations, quotes), while regex is simpler but may split incorrectly.


### RAG Pipeline Modular Coding

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_classic.schema import Document
from langchain_classic.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [6]:
### Custom Semantic Chunker With Threshold

class ThresholdSemanticChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(model_name)
        self.threshold=threshold 

    def split(self, text: str):
        import nltk
        nltk.download('punkt', quiet=True)  # Ensure punkt is downloaded
        sentences = nltk.sent_tokenize(text.strip())
        sentences = [s.strip() for s in sentences if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]

        chunks.append(" ".join(current_chunk))
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result

In [7]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [8]:
### Chunking
chunker=ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [9]:
### VectorStore
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
embedding=OpenAIEmbeddings()
vectorstore=FAISS.from_documents(chunks,embedding)
retriever=vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C3C78FEC90>, search_kwargs={})

In [11]:
## Prompt Template

# --- 5. Prompt Template ---
template = """Answer the question based on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n{context}\n\nQuestion: {question}\n')

In [14]:
## LLM
llm=init_chat_model(model="groq:llama-3.1-8b-instant",temperature=0.4)

### LCEL Chain With retrieval

rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 8. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

LangChain is a framework for building applications with LLMs (Large Language Models). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


### Semantic chunker With Langchain

In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_classic.document_loaders import TextLoader

In [16]:
## Load the documents
loader=TextLoader("langchain_intro.txt")
docs=loader.load()

## Initialize embedding model
embedding=OpenAIEmbeddings()

## Create the semantic chunker
chunker=SemanticChunker(embedding)

## Split the documents
chunks=chunker.split_documents(docs)

## Result

for i,chunk in enumerate(chunks):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")


 chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

 chunk 2:
You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.
